# Traffic Analyzer – Interactive Demo

This notebook demonstrates the key capabilities of the **Traffic Analyzer** powered by Agentic AI with Groq and LangChain.

## Setup

Before running this notebook, make sure you have:
1. Installed dependencies: `pip install -r requirements.txt`
2. Created a `.env` file with your Groq API key


In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root to path

from dotenv import load_dotenv
load_dotenv('../.env')

print('Environment loaded ✓')

## 1. Traffic Conditions

Get current traffic conditions for a location.

In [ ]:
from src.services.traffic import TrafficService
import json

traffic_svc = TrafficService()

# Get traffic conditions for New York City
conditions = traffic_svc.get_traffic_conditions('New York City', '40.7128,-74.0060')
analysis = traffic_svc.analyze_congestion(conditions)

print('=== Traffic Conditions: New York City ===')
print(f"Congestion Score : {conditions['congestion_score']}/100")
print(f"Average Speed    : {conditions['average_speed_kmh']} km/h")
print(f"Incidents        : {conditions['incident_count']}")
print(f"Road Conditions  : {conditions['road_conditions']}")
print()
print(f"Congestion Level : {analysis['level'].upper()}")
print(f"Description      : {analysis['description']}")
print(f"Recommendation   : {analysis['recommendation']}")

## 2. Weather Impact

Assess how weather affects driving conditions.

In [ ]:
from src.services.weather import WeatherService

weather_svc = WeatherService()

weather = weather_svc.get_current_weather('New York City')
impact = weather_svc.assess_traffic_impact(weather)

print('=== Weather Conditions ===')
print(f"Condition        : {weather['condition'].title()}")
print(f"Temperature      : {weather['temperature_c']} °C")
print(f"Wind Speed       : {weather['wind_speed_kmh']} km/h")
print(f"Visibility       : {weather['visibility_km']} km")
print(f"Precipitation    : {weather['precipitation_mm']} mm")
print()
print('=== Traffic Impact ===')
print(f"Severity         : {impact['severity'].upper()}")
print(f"Delay Multiplier : ×{impact['delay_multiplier']}")
print(f"Recommendation   : {impact['recommendation']}")

## 3. Route Optimisation

Find the best route between two locations.

In [ ]:
from src.services.route import RouteService

route_svc = RouteService()

# Get congestion for start location
congestion_score = conditions['congestion_score']
delay_mult = impact['delay_multiplier']

routes = route_svc.get_alternative_routes(
    start='New York City',
    end='Philadelphia',
    congestion_score=congestion_score,
    weather_delay_multiplier=delay_mult,
)

print('=== Route Options: New York → Philadelphia ===')
for route in routes['routes']:
    label = '✅ RECOMMENDED' if route.get('rank') == 1 else f"  Option {route.get('rank')}"
    rt = route.get('route_type', 'direct')
    print(f"{label} [{rt}]  {route['distance_km']} km  |  {route['estimated_duration_min']} min")

## 4. Congestion Prediction

Predict future congestion levels.

In [ ]:
predictions = traffic_svc.predict_congestion('New York City', hours_ahead=6)

print('=== Congestion Predictions: Next 6 Hours ===')
print(f"{'Hours':<8} {'UTC Hour':<12} {'Score':<8} {'Level'}")
print('-' * 40)
for pred in predictions['predictions']:
    print(f"{pred['hours_from_now']:<8} {pred['predicted_hour']:02d}:00{' '*6} {pred['congestion_score']:<8} {pred['level'].upper()}")

## 5. Agentic AI Analysis

Use the Groq-powered agent for intelligent, multi-step analysis.

> **Note**: Requires a valid `GROQ_API_KEY` in your `.env` file.

In [ ]:
import os

groq_key = os.getenv('GROQ_API_KEY', '')

if not groq_key:
    print('⚠️  GROQ_API_KEY not set. Skipping agent demo.')
    print('   Add your API key to .env to enable this section.')
else:
    from src.agent.groq_agent import TrafficAnalyzerAgent

    agent = TrafficAnalyzerAgent(api_key=groq_key)

    print('🤖 Agent query: Analyze traffic in New York and suggest the best time to travel to Philadelphia')
    result = agent.analyze(
        'What are the current traffic conditions in New York City? '
        'Should I drive to Philadelphia now or wait? '
        'Check weather impact too.'
    )
    print()
    print(result['output'])

## 6. REST API Demo

Interact with the API using the `httpx` client.

> **Note**: Start the API server first with `uvicorn api.main:app --reload`.

In [ ]:
import httpx

BASE_URL = 'http://localhost:8000'

try:
    with httpx.Client(timeout=10) as client:
        # Health check
        resp = client.get(f'{BASE_URL}/health')
        print('Health:', resp.json())

        # Analyze traffic
        resp = client.post(
            f'{BASE_URL}/api/v1/analyze',
            json={'location': 'New York', 'include_weather': True},
        )
        data = resp.json()
        print(f"\nCongestion Score : {data['traffic']['congestion_score']}/100")
        print(f"Level            : {data['congestion_analysis']['level'].upper()}")
except httpx.ConnectError:
    print('⚠️  Could not connect to API. Start the server with: uvicorn api.main:app --reload')